# EDA — Dataset completo (dengue clásico + grave) · 2007–2024

Análisis del dataset unificado generado en `04_Combinar_Ambos_Datasets.ipynb`. Compara la distribución temporal y geográfica entre ambos tipos, identifica los municipios endémicos candidatos al modelo y genera las tablas base para el corredor endémico.

**Prerrequisito:** haber corrido los notebooks 02, 03 y 04.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4),
                     'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
CSV = "../data/processed/sivigila_dengue_completo.csv"

# Años de epidemia mayor — excluidos del período de referencia del corredor endémico
EPIDEMIC_YEARS = [2010, 2013, 2016, 2019, 2023, 2024]
REF_START, REF_END = 2007, 2022   # período de referencia para corredor endémico

# Criterio de municipio endémico (D4)
MIN_YEARS  = 10   # ≥10 de 18 años con casos
MIN_CASES  = 200  # ≥200 casos acumulados 2007–2024

## 1. Carga

In [ ]:
df = pd.read_csv(CSV, dtype=str, low_memory=False)
df['ANO']    = pd.to_numeric(df['ANO'],    errors='coerce')
df['SEMANA'] = pd.to_numeric(df['SEMANA'], errors='coerce')
print(f"Total: {len(df):,} filas  |  {df.shape[1]} columnas")
print("\nDistribución por tipo:")
display(df['tipo_dengue'].value_counts().to_frame())

## 2. Serie temporal — clásico vs grave por año

In [ ]:
por_ano_tipo = (
    df.groupby(['ANO', 'tipo_dengue'])
    .size()
    .unstack(fill_value=0)
    .rename_axis('Año')
)

fig, ax = plt.subplots()
width = 0.4
x = np.arange(len(por_ano_tipo))
ax.bar(x - width/2, por_ano_tipo.get('clasico', 0), width, label='Clásico', color='#5C8DBE')
ax.bar(x + width/2, por_ano_tipo.get('grave',   0), width, label='Grave',   color='#BE1D2B')
ax.set_xticks(x)
ax.set_xticklabels(por_ano_tipo.index, rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title('Casos de dengue clásico vs grave por año (2007–2024)')
ax.legend()
plt.tight_layout()
plt.show()

por_ano_tipo['total'] = por_ano_tipo.sum(axis=1)
por_ano_tipo['% grave'] = (por_ano_tipo.get('grave', 0) / por_ano_tipo['total'] * 100).round(2)
display(por_ano_tipo.style.format({c: '{:,}' for c in ['clasico', 'grave', 'total']}
                                  | {'% grave': '{:.2f}%'}))

## 3. Estacionalidad combinada (semana epidemiológica)

In [ ]:
df_ref = df[(df['ANO'] >= REF_START) & (df['ANO'] <= REF_END)
            & (~df['ANO'].isin(EPIDEMIC_YEARS))]

sem_tipo = (
    df_ref.groupby(['SEMANA', 'tipo_dengue'])
    .size()
    .unstack(fill_value=0)
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
sem_tipo.get('clasico', pd.Series()).plot.bar(ax=axes[0], color='#5C8DBE', width=0.8)
sem_tipo.get('grave',   pd.Series()).plot.bar(ax=axes[1], color='#BE1D2B', width=0.8)
axes[0].set_title('Estacionalidad — dengue clásico (años no epidémicos)')
axes[1].set_title('Estacionalidad — dengue grave (años no epidémicos)')
axes[1].set_xlabel('Semana epidemiológica')
for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
plt.tight_layout()
plt.show()

## 4. Municipios endémicos (criterio D4)

In [ ]:
# Solo dengue clásico para el criterio endémico
df_c = df[df['tipo_dengue'] == 'clasico'].copy()

# Años con al menos 1 caso por municipio
anos_con_casos = (
    df_c.groupby(['Municipio_ocurrencia', 'COD_MUN_O', 'ANO'])
    .size()
    .gt(0)
    .groupby(level=[0, 1])
    .sum()
    .rename('anos_con_casos')
)

# Casos acumulados
casos_acum = (
    df_c.groupby(['Municipio_ocurrencia', 'COD_MUN_O'])
    .size()
    .rename('casos_acumulados')
)

endemicos = pd.concat([anos_con_casos, casos_acum], axis=1).reset_index()
endemicos = endemicos[
    (endemicos['anos_con_casos'] >= MIN_YEARS) &
    (endemicos['casos_acumulados'] >= MIN_CASES)
].sort_values('casos_acumulados', ascending=False)

print(f"Municipios endémicos (≥{MIN_YEARS} años y ≥{MIN_CASES} casos): {len(endemicos):,}")
display(endemicos.head(20).style.format({'anos_con_casos': '{:,}', 'casos_acumulados': '{:,}'}))

In [ ]:
# Guardar lista de municipios endémicos
OUT_ENDEMICOS = "../data/processed/municipios_endemicos.csv"
os.makedirs(os.path.dirname(os.path.abspath(OUT_ENDEMICOS)), exist_ok=True)
endemicos.to_csv(OUT_ENDEMICOS, index=False, encoding='utf-8')
print(f"Guardado: {os.path.abspath(OUT_ENDEMICOS)}")

## 5. Corredor endémico nacional — P25 / Mediana / P75

In [ ]:
df_ref_c = df_c[(df_c['ANO'] >= REF_START) & (df_c['ANO'] <= REF_END)
                & (~df_c['ANO'].isin(EPIDEMIC_YEARS))]

semanal = df_ref_c.groupby(['ANO', 'SEMANA']).size().reset_index(name='casos')
corredor = semanal.groupby('SEMANA')['casos'].agg(
    p25=lambda x: x.quantile(0.25),
    mediana='median',
    p75=lambda x: x.quantile(0.75)
).reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(corredor['SEMANA'], corredor['p25'],  corredor['p75'],
                alpha=0.25, color='#1A7F37', label='Zona endémica (P25–P75)')
ax.fill_between(corredor['SEMANA'], corredor['p25'],  corredor['mediana'],
                alpha=0.35, color='#1A7F37')
ax.plot(corredor['SEMANA'], corredor['mediana'], color='#1A7F37', lw=2, label='Mediana')
ax.plot(corredor['SEMANA'], corredor['p75'],     color='#C27800', lw=1.5,
        linestyle='--', label='P75 (umbral epidémico)')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos')
ax.set_title(f'Corredor endémico nacional — período de referencia {REF_START}–{REF_END}')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Correlación temporal clásico vs grave (semanal, por año)

In [ ]:
semanal_c = (
    df[df['tipo_dengue'] == 'clasico']
    .groupby(['ANO', 'SEMANA']).size().rename('clasico')
)
semanal_g = (
    df[df['tipo_dengue'] == 'grave']
    .groupby(['ANO', 'SEMANA']).size().rename('grave')
)
semanal_ambos = pd.concat([semanal_c, semanal_g], axis=1).fillna(0)

corr = semanal_ambos.corr().loc['clasico', 'grave']
print(f"Correlación semanal clásico–grave: r = {corr:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(semanal_ambos['clasico'], semanal_ambos['grave'],
           alpha=0.15, s=8, color='#BE1D2B')
m, b = np.polyfit(semanal_ambos['clasico'], semanal_ambos['grave'], 1)
x_line = np.linspace(0, semanal_ambos['clasico'].max(), 100)
ax.plot(x_line, m * x_line + b, color='#1B2233', lw=1.5, label=f'Ajuste lineal (r={corr:.2f})')
ax.set_xlabel('Casos clásico (semanal)')
ax.set_ylabel('Casos graves (semanal)')
ax.set_title('Correlación casos clásico vs grave por semana epidemiológica')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Exportar tabla semanal por municipio

In [ ]:
# Tabla base para el modelo: municipio × semana × tipo
tabla_semanal = (
    df.groupby(['COD_MUN_O', 'Municipio_ocurrencia', 'Departamento_ocurrencia',
                'ANO', 'SEMANA', 'tipo_dengue'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
tabla_semanal.columns.name = None

OUT_SEMANAL = "../data/processed/dengue_semanal_municipio_completo.csv"
tabla_semanal.to_csv(OUT_SEMANAL, index=False, encoding='utf-8')
print(f"Filas: {len(tabla_semanal):,}")
print(f"Guardado: {os.path.abspath(OUT_SEMANAL)}")
display(tabla_semanal.head())